# Exercise 8: RLHF Optimization — PPO-KL and DPO

---

## Learning Objectives

By the end of this exercise, you will be able to:

1. **Fine-tune a language model** with Supervised Fine-Tuning (SFT) as the RLHF starting point
2. **Implement PPO with KL control** for RLHF-style policy optimization on a real LLM
3. **Derive and implement the DPO loss** from the Bradley-Terry preference model
4. **Compare all methods** on sentiment-controlled text generation

---

## Outline

| Part | Topic | Time |
|------|-------|------|
| 0 | Model, Tokenizer & Reward Signal | 5 min |
| 1 | Supervised Fine-Tuning (Stage I) | 10 min |
| 2 | PPO with KL Control (Stage III) | 20 min |
| 3 | Direct Preference Optimization (DPO) | 15 min |
| 4 | Comparative Evaluation & Discussion | 10 min |

## Setup

Run the cell below to install dependencies and import libraries.

> **VRAM budget:** This notebook uses **GPT-2** (124 M params, ~500 MB) with **LoRA** (~300 K trainable params) and a **DistilBERT** sentiment classifier (~260 MB). Total GPU memory ≈ **1–1.5 GB**. You can choose to use Colab, Kaggle or even your own laptop's GPU if it has a suitable one.

In [ ]:
import subprocess, sys
for pkg in ["transformers", "peft", "datasets"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import copy
from typing import Dict, Tuple, List
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

---

## Part 0: Model, Tokenizer & Reward Signal (5 min)

### The Three Stages of RLHF

| Stage | Description |
|-------|-------------|
| **I. SFT** | Supervised fine-tuning on demonstration data |
| **II. Reward Model** | Train $r_\phi(x, y)$ on preference pairs $(y_w \succ y_l \mid x)$ |
| **III. Policy Optimization** | Optimize $\pi_\theta$ against the reward model while staying close to the SFT reference |

We use **GPT-2** as our base language model — small enough to train on a single consumer GPU yet large enough to generate coherent text. For the reward signal we use a pre-trained **DistilBERT** sentiment classifier on IMDB, acting as a lightweight reward model that scores how "positive" a generated movie review is.

This lets us run the full RLHF pipeline (SFT → reward → PPO / DPO) in minutes.

In [ ]:
MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.config.pad_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {n_params:,} (~{n_params * 4 / 1e6:.0f} MB in fp32)")

In [ ]:
REWARD_MODEL_NAME = "lvwerra/distilbert-imdb"

reward_tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_NAME)
reward_net = AutoModelForSequenceClassification.from_pretrained(REWARD_MODEL_NAME).to(device)
reward_net.eval()
for p in reward_net.parameters():
    p.requires_grad = False

@torch.no_grad()
def compute_sentiment_reward(texts: List[str]) -> torch.Tensor:
    """Return sentiment reward in [-1, +1] (centred at 0).

    Maps P(POSITIVE) via  r = 2*p - 1  so that neutral text scores ~0,
    positive text scores up to +1, and negative text down to -1.
    This wider, zero-centred range gives RL a much clearer gradient signal.
    """
    enc = reward_tokenizer(
        texts, padding=True, truncation=True, max_length=512, return_tensors="pt",
    ).to(device)
    logits = reward_net(**enc).logits
    p_pos = F.softmax(logits, dim=-1)[:, 1]
    return 2.0 * p_pos - 1.0

# Sanity check
for text, label in [
    ("This movie was absolutely fantastic! A true masterpiece of cinema.", "Positive"),
    ("Terrible film. A complete waste of time. I hated every scene.", "Negative"),
    ("The weather today is partly cloudy with a chance of rain.", "Neutral"),
]:
    r = compute_sentiment_reward([text]).item()
    print(f"  {label:>8s}: reward = {r:+.3f}  |  {text[:60]}")

In [ ]:
PROMPT_LEN = 16
MAX_NEW_TOKENS = 48

dataset = load_dataset("imdb", split="train")
positive_texts = [x["text"] for x in dataset if x["label"] == 1][:500]
all_texts = [x["text"] for x in dataset]

# Build a set of fixed-length prompts (first PROMPT_LEN tokens of reviews)
np.random.seed(SEED)
shuffled_idx = np.random.permutation(len(all_texts))
prompts = []
for i in shuffled_idx:
    ids = tokenizer.encode(all_texts[i], add_special_tokens=False)[:PROMPT_LEN]
    if len(ids) == PROMPT_LEN:
        prompts.append(tokenizer.decode(ids))
    if len(prompts) >= 200:
        break

print(f"SFT examples: {len(positive_texts)}")
print(f"RL prompts:   {len(prompts)}")
print(f"Sample prompt: \"{prompts[0]}\"")

# ── Helper functions ──────────────────────────────────────────

def encode_prompts(prompt_list: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
    """Encode a list of prompt strings to fixed-length, left-padded tensors."""
    enc = tokenizer(
        prompt_list, return_tensors="pt",
        padding="max_length", truncation=True, max_length=PROMPT_LEN,
    ).to(device)
    return enc.input_ids, enc.attention_mask

@torch.no_grad()
def generate_responses(gen_model, prompt_ids, prompt_mask):
    """Generate exactly MAX_NEW_TOKENS tokens per prompt."""
    return gen_model.generate(
        input_ids=prompt_ids,
        attention_mask=prompt_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        min_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.pad_token_id,
    )

def make_full_mask(prompt_mask, batch_size):
    """Extend prompt attention mask to cover generated tokens."""
    return torch.cat([
        prompt_mask,
        torch.ones(batch_size, MAX_NEW_TOKENS, device=device, dtype=torch.long),
    ], dim=1)

def get_response_log_probs(lm, input_ids, attention_mask):
    """Compute per-token and total log-probs for response tokens.

    Prompt occupies positions [0, PROMPT_LEN); response occupies [PROMPT_LEN, T).
    """
    logits = lm(input_ids=input_ids, attention_mask=attention_mask).logits
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    token_lps = log_probs.gather(2, input_ids[:, 1:].unsqueeze(2)).squeeze(2)

    mask = torch.zeros_like(token_lps)
    mask[:, PROMPT_LEN - 1 :] = attention_mask[:, PROMPT_LEN:].float()

    per_token = token_lps * mask
    total = per_token.sum(dim=-1)
    return total, per_token, mask

def decode_texts(full_ids):
    return tokenizer.batch_decode(full_ids, skip_special_tokens=True)

---

## Part 1: Supervised Fine-Tuning — Stage I (10 min)

### Background

Before any reward-based optimization, the base language model is fine-tuned on **demonstration data** — high-quality examples of the desired behaviour. This produces the **SFT model** $\pi_{\text{SFT}}$, which serves two roles:

1. **Starting point** for Stage III policy optimization.
2. **Reference policy** $\pi_{\text{ref}}$ — the KL penalty keeps the RL-trained policy close to this model, preventing reward hacking.

For our sentiment-controlled generation task, we fine-tune GPT-2 on **positive IMDB reviews** so that it learns the domain and already has a mild positive-sentiment bias.

## 🔴 Exercise 1: Implement SFT

Implement an SFT training loop using the standard causal language modelling loss **with LoRA**.

Full fine-tuning of all 124 M parameters on a small dataset (500 reviews) causes **catastrophic forgetting** — the model overfits to the narrow training distribution and loses its general generation ability. LoRA keeps the base weights frozen and only trains a lightweight adapter, preserving the model's fluency while shifting it toward positive sentiment.

---

### What is a LoRA Adapter?

**LoRA** (Low-Rank Adaptation, Hu et al. 2022) avoids updating full weight matrices by injecting a low-rank decomposition alongside them.

For a frozen pre-trained weight matrix $W_0 \in \mathbb{R}^{d \times k}$, LoRA adds:

$$W = W_0 + \frac{\alpha}{r} \cdot BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ are the **trainable adapter matrices**, and $r \ll \min(d, k)$ is the **rank**. $W_0$ is frozen throughout training.

- **Initialisation:** $A$ is drawn from $\mathcal{N}(0, \sigma^2)$; $B$ is initialised to zero. This ensures $BA = 0$ at the start, so training begins from the pre-trained model.
- **Where it is applied:** typically to the query/key/value projection matrices inside each attention block (and optionally the output projection).

### How the Adapter is Merged

After training, the adapter can be **fused** back into the base weight with a single matrix addition:

$$W_{\text{merged}} = W_0 + \frac{\alpha}{r} \cdot BA$$

The result is a standard weight matrix — no extra computation or memory at inference time. In PEFT this is done with:

```python
model = model.merge_and_unload()
```

This is exactly what you should do at the end of `train_sft`. The returned model is a plain `AutoModelForCausalLM` with the LoRA updates baked in, ready to be used as the SFT reference policy.

---


> **💭 Theory Exercise — Parameter Count**
>
> Full fine-tuning of $W_0 \in \mathbb{R}^{d \times k}$ trains all $d \times k$ parameters.
> LoRA instead trains only $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$.
>
> 1. Write the formula for the number of trainable parameters introduced by one LoRA adapter.
> 2. GPT-2's combined QKV projection has $d = k = 768$. With rank $r = 16$, compute: (a) trainable params with full fine-tuning, (b) trainable params with LoRA, (c) the reduction factor.

**Answer** :


### Configuring the LoRA Adapter

LoRA is configured via `LoraConfig`. The key parameters:

| Parameter | What it controls | Typical value |
|-----------|-----------------|---------------|
| `r` | Rank of $A$ and $B$. Higher = more capacity, more params. | 8–64 |
| `lora_alpha` | Scaling factor $\alpha$. The effective learning rate of the adapter scales as $\alpha / r$. Often set to $2r$ or $r$. | 16–64 |
| `target_modules` | Which weight matrices to adapt. GPT-2 uses `"c_attn"` (combined QKV projection) and `"c_proj"` (output projection). | model-dependent |
| `lora_dropout` | Dropout applied to adapter activations — regularises small adapters. | 0.0–0.1 |
| `bias` | Whether to train bias terms alongside adapters. `"none"` keeps biases frozen. | `"none"` |
| `task_type` | Tells PEFT how to wrap the model (affects how inputs/outputs are handled). | `"CAUSAL_LM"` |


Choosing `target_modules` requires knowing the model's architecture. You can inspect all named modules with:

```python
for name, module in model.named_modules():
    print(name, type(module).__name__)
```

In the next exercise you can set 
```python
r=16, 
lora_alpha=32,
target_modules=["c_attn", "c_proj"],
lora_dropout=0.05,
bias="none",
task_type="CAUSAL_LM
```

---

**Your task:** Complete the `train_sft` function below.

**Hint:** Use the model's built-in loss by passing `labels=input_ids` (with padding positions set to $-100$). After training, merge the LoRA adapter into the base weights with `merge_and_unload()`.

In [ ]:
def evaluate_model(eval_model, eval_prompts, n=16, label="Model"):
    """Generate responses and measure mean sentiment reward."""
    p_ids, p_mask = encode_prompts(eval_prompts[:n])
    eval_model.eval()
    full_ids = generate_responses(eval_model, p_ids, p_mask)
    texts = decode_texts(full_ids)
    rewards = compute_sentiment_reward(texts)
    print(f"\n{'='*60}")
    print(f"  {label} | mean sentiment reward: {rewards.mean():+.3f}")
    print(f"{'='*60}")
    for i in range(min(3, n)):
        print(f"  [r={rewards[i]:+.3f}] {texts[i][:180]}...")
    return rewards.mean().item()


# ── Evaluate the base (pre-trained) model before any fine-tuning ──
base_reward = evaluate_model(model, prompts, label="Base GPT-2 (pre-trained)")


def train_sft(base_model, texts, n_steps=300, batch_size=4, lr=2e-4, max_length=128):
    """Fine-tune a causal LM on demonstration data (RLHF Stage I).

    Uses LoRA so that the base model's general language capabilities are
    preserved — full fine-tuning on a small dataset causes catastrophic
    forgetting.  After training the adapter is merged into the base weights.
    """
    # ============================================================
    # 🔴 TODO: Implement the SFT training loop
    #
    # Steps:
    # 1. Attach a LoRA adapter to the base model
    # 2. Tokenize all texts (padding + truncation)
    # 3. For each step, sample a random mini-batch
    # 4. Forward pass: use labels = input_ids, masking pad tokens to -100
    # 5. Backpropagate and update (only LoRA parameters)
    # 6. Merge the trained adapter back into the base weights
    # ============================================================


sft_history, model = train_sft(model, positive_texts)

plt.plot(sft_history["loss"], alpha=0.3, color="steelblue")
w = 20
sm = np.convolve(sft_history["loss"], np.ones(w) / w, mode="valid")
plt.plot(range(w - 1, len(sft_history["loss"])), sm, color="steelblue", lw=2)
plt.xlabel("Step"); plt.ylabel("Loss"); plt.title("SFT Training Loss")
plt.show()

In [ ]:
# Save SFT weights — this is our frozen reference for all RL methods.
# model now holds the merged SFT weights (base + LoRA baked in).
sft_state = copy.deepcopy(model.state_dict())
sft_reward = evaluate_model(model, prompts, label="SFT model")

print(f"\nImprovement from SFT over base: {sft_reward - base_reward:+.3f} reward")

In [ ]:
def create_rl_model():
    """Load a fresh GPT-2, restore SFT weights, and attach a LoRA adapter.

    The base (non-LoRA) weights are the SFT/reference model.
    Calling ``model.disable_adapter()`` gives reference-policy outputs.
    """
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    m.config.pad_token_id = tokenizer.pad_token_id
    m.load_state_dict(sft_state)
    lora_cfg = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["c_attn", "c_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    m = get_peft_model(m, lora_cfg)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    print(f"  LoRA trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return m

---

## Part 2: PPO with KL Control — Stage III (20 min)

### The RLHF Objective

Given a frozen reward model $r_\phi$ and a reference policy $\pi_{\text{ref}}$ (the SFT model), Stage III solves:

$$\max_{\pi_\theta} \; \mathbb{E}_{x \sim \mathcal{D},\, y \sim \pi_\theta(\cdot|x)} \big[ r_\phi(x, y) \big] - \beta \, \text{KL}\!\left[ \pi_\theta(\cdot|x) \,\|\, \pi_{\text{ref}}(\cdot|x) \right]$$

The KL term prevents **reward hacking** — the policy diverging from the SFT model to exploit imperfections in the learned reward model.

### PPO-KL Objective

PPO adapts the clipped surrogate from Schulman et al. (2017). For a single sample $(x, y)$:

$$L^{\text{CLIP}}(\theta) = \mathbb{E} \left[ \min\!\left( \rho_t A_t, \; \text{clip}(\rho_t, 1-\epsilon, 1+\epsilon) \, A_t \right) \right]$$

where $\rho_t = \frac{\pi_\theta(y|x)}{\pi_{\theta_{\text{old}}}(y|x)}$ is the importance ratio. In the RLHF variant, the advantage incorporates the KL penalty:

$$A = r_\phi(x, y) - \beta \, \text{KL}[\pi_\theta \| \pi_{\text{ref}}] \;-\; \text{baseline}$$

### Token-Level Details

For autoregressive models, the per-sample log-probability is the sum of per-token log-probs:

$$\log \pi_\theta(y|x) = \sum_{t=1}^{T} \log \pi_\theta(y_t \,|\, x, y_{<t})$$

and the KL divergence estimate for a single sample is:

$$\widehat{\text{KL}}_i = \log \pi_\theta(y_i|x_i) - \log \pi_{\text{ref}}(y_i|x_i)$$

## 🔴 Exercise 2: Implement the PPO-KL Update Step

**Your task:** Complete the `ppo_kl_step` function.

**Important — per-token ratios:** For autoregressive models the sequence-level ratio $\prod_t \rho_t$ can explode (or collapse to 0) because 48 per-token ratios are multiplied together. The standard implementation instead computes **per-token** ratios $\rho_t = \exp(\log\pi_t^{\text{curr}} - \log\pi_t^{\text{old}})$, keeps them individually clipped, and assigns the same *sequence-level* advantage to every response token.

**Steps:**
1. Per-token current & reference log-probs
2. Mean-per-token KL per sequence → KL-penalised reward → normalised advantage
3. Broadcast that advantage to every response token
4. Per-token importance ratio $\rho_t$
5. Per-token clipped surrogate, averaged over all response tokens

In [ ]:
def ppo_kl_step(ppo_model, full_ids, full_mask, old_per_token_lps, rewards,
                beta=0.02, clip_eps=0.2):
    """Compute the PPO-KL loss for one optimisation step.

    **Per-token computation:** importance ratios are computed per token so
    that clipping works correctly.  A sequence-level advantage (from the
    KL-penalised reward) is broadcast to every response token.

    Args:
        ppo_model:         PEFT policy (LoRA ON = current, OFF = reference)
        full_ids:          [B, T]   generated sequences
        full_mask:         [B, T]   attention mask
        old_per_token_lps: [B, T-1] per-token log-probs from sampling policy
        rewards:           [B]      sentiment rewards
        beta:              per-token KL penalty coefficient
        clip_eps:          PPO clipping range

    Returns:
        loss, info
    """
    # ============================================================
    # 🔴 TODO: Implement the PPO-KL loss (per-token version)
    #
    # 1. Per-token current & reference log-probs
    # 2. Per-token KL  →  mean-per-token KL per sequence
    # 3. KL-penalised reward  →  normalised sequence-level advantage
    # 4. Broadcast advantage to each response token
    # 5. Per-token importance ratio  =  exp(curr_t − old_t)
    # 6. Per-token clipped surrogate, averaged over response tokens
    # ============================================================

    

### Training PPO-KL

In [ ]:
def train_ppo_kl(n_iters=150, batch_size=8, ppo_epochs=3,
                 lr=5e-5, beta=0.02, clip_eps=0.2):
    """Full PPO-KL training loop with a real LLM."""
    ppo_model = create_rl_model()
    optimizer = torch.optim.AdamW(
        (p for p in ppo_model.parameters() if p.requires_grad), lr=lr,
    )
    history = {"reward": [], "kl": []}

    for it in range(n_iters):
        idx = np.random.choice(len(prompts), batch_size, replace=False)
        batch_prompts = [prompts[i] for i in idx]
        p_ids, p_mask = encode_prompts(batch_prompts)

        ppo_model.eval()
        full_ids = generate_responses(ppo_model, p_ids, p_mask)
        full_mask = make_full_mask(p_mask, batch_size)
        rewards = compute_sentiment_reward(decode_texts(full_ids))

        with torch.no_grad():
            _, old_pt_lps, _ = get_response_log_probs(ppo_model, full_ids, full_mask)

        ppo_model.train()
        for _ in range(ppo_epochs):
            loss, info = ppo_kl_step(
                ppo_model, full_ids, full_mask, old_pt_lps, rewards,
                beta=beta, clip_eps=clip_eps,
            )
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ppo_model.parameters(), 1.0)
            optimizer.step()

        history["reward"].append(info["reward"])
        history["kl"].append(info["kl"])

        if (it + 1) % 30 == 0:
            print(f"  Iter {it+1:3d}/{n_iters} | reward {info['reward']:+.3f} "
                  f"| KL {info['kl']:.4f} | clip {info['clip_frac']:.0%}")

    return history, ppo_model


ppo_history, ppo_model = train_ppo_kl()
ppo_reward = evaluate_model(ppo_model, prompts, label="PPO-KL")

In [ ]:
def plot_rl_training(history, title, keys=None):
    keys = keys or [("reward", "Mean Reward", "steelblue"),
                    ("kl", "KL from Reference", "coral")]
    fig, axes = plt.subplots(1, len(keys), figsize=(7 * len(keys), 4))
    if len(keys) == 1:
        axes = [axes]
    w = 10
    for ax, (k, label, color) in zip(axes, keys):
        data = np.array(history[k])
        ax.plot(data, alpha=0.25, color=color)
        if len(data) >= w:
            sm = np.convolve(data, np.ones(w) / w, mode="valid")
            ax.plot(range(w - 1, len(data)), sm, color=color, lw=2)
        ax.set_xlabel("Iteration"); ax.set_ylabel(label); ax.set_title(label)
    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

plot_rl_training(ppo_history, "PPO-KL Training")

### 💭 Discussion Question

**Question:** What happens if you set $\beta = 0$ (no KL penalty)? What about $\beta = 2$ (very large)?


**Answer:** 

---

## Part 3: Direct Preference Optimization — DPO (15 min)

### Motivation

PPO-KL is powerful but operationally complex:
1. It requires a **separate reward model** in GPU memory.
2. It performs **online generation** every iteration (slow).
3. PPO has many sensitive hyperparameters (clip ratio, epochs, KL coefficient, …).

**DPO** (Rafailov et al., 2023) eliminates the reward model by showing that the optimal policy under the KL-constrained objective has a closed-form relationship with the reward.

### Derivation

**Step 1 — The optimal policy** of $\max_\pi \mathbb{E}[r(x,y)] - \beta\,\text{KL}[\pi\|\pi_{\text{ref}}]$ is:

$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x)\,\exp\!\left(\frac{r(x,y)}{\beta}\right)$$

**Step 2 — Invert** to express the reward in terms of the policy:

$$r(x,y) = \beta\log\frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta\log Z(x)$$

**Step 3 — Substitute into the Bradley-Terry preference model** $p(y_w \succ y_l|x) = \sigma(r(x,y_w)-r(x,y_l))$. The partition functions $Z(x)$ cancel:

$$p(y_w \succ y_l|x) = \sigma\!\left(\beta\log\frac{\pi^*(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta\log\frac{\pi^*(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$$

**The DPO Loss** — Replace $\pi^*$ with the trainable $\pi_\theta$ and maximise the log-likelihood:

$$\boxed{\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x,y_w,y_l)}\!\left[\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]}$$

**Key insight:** DPO optimises the same objective as RLHF but trains **directly on preference data** — no reward model, no online sampling.

### Generating a Preference Dataset

For DPO we need pairs $(x, y_w, y_l)$. We generate two completions per prompt from the **SFT model**, score them with the sentiment classifier, and label the higher-scoring one as *chosen*.

In [ ]:
@torch.no_grad()
def generate_preference_dataset(ref_model, n_pairs=400, batch_size=16):
    """Create preference pairs from the SFT model."""
    ref_model.eval()
    data = {"chosen_ids": [], "rejected_ids": [], "mask": []}

    for start in range(0, n_pairs, batch_size):
        size = min(batch_size, n_pairs - start)
        idx = np.random.choice(len(prompts), size, replace=False)
        p_ids, p_mask = encode_prompts([prompts[i] for i in idx])

        resp_a = generate_responses(ref_model, p_ids, p_mask)
        resp_b = generate_responses(ref_model, p_ids, p_mask)
        ext_mask = make_full_mask(p_mask, size)

        r_a = compute_sentiment_reward(decode_texts(resp_a))
        r_b = compute_sentiment_reward(decode_texts(resp_b))

        for j in range(size):
            if r_a[j] >= r_b[j]:
                data["chosen_ids"].append(resp_a[j])
                data["rejected_ids"].append(resp_b[j])
            else:
                data["chosen_ids"].append(resp_b[j])
                data["rejected_ids"].append(resp_a[j])
            data["mask"].append(ext_mask[j])

        if (start + batch_size) % 80 == 0:
            print(f"  {start + size}/{n_pairs} pairs generated")

    return {k: torch.stack(v) for k, v in data.items()}


# Build a fresh SFT model just for generation (saves memory vs keeping ppo_model)
pref_gen_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
pref_gen_model.config.pad_token_id = tokenizer.pad_token_id
pref_gen_model.load_state_dict(sft_state)

print("Generating preference dataset from SFT model...")
pref_data = generate_preference_dataset(pref_gen_model, n_pairs=400)
del pref_gen_model; torch.cuda.empty_cache()
print(f"Done — {len(pref_data['chosen_ids'])} preference pairs")

## 🔴 Exercise 3: Implement the DPO Loss

**Your task:** Complete the `dpo_loss` function.

**Hint:** Compute log-ratios $\log\frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$ for both chosen and rejected, then apply the Bradley-Terry formula.

In [ ]:
def dpo_loss(dpo_model, chosen_ids, rejected_ids, mask, beta=0.1):
    """Compute the DPO loss on a batch of preference pairs.

    Args:
        dpo_model:    PEFT policy (disable adapter → reference)
        chosen_ids:   [B, T] — preferred completions
        rejected_ids: [B, T] — dis-preferred completions
        mask:         [B, T] — attention mask (shared, same prompt lengths)
        beta:         temperature controlling deviation from reference

    Returns:
        loss, info
    """
    # ============================================================
    # 🔴 TODO: Implement the DPO loss
    #
    # 1. Policy log-probs for chosen and rejected
    # 2. Reference log-probs (disable LoRA)
    # 3. Log-ratios
    # 4. DPO logits = beta * (log_ratio_chosen - log_ratio_rejected)
    # 5. Loss = -log(sigmoid(logits)).mean()
    # ============================================================


### Training DPO

In [ ]:
def train_dpo(pref_data, n_steps=300, batch_size=4, lr=5e-5, beta=0.1):
    """DPO training loop — offline, no generation needed."""
    dpo_model = create_rl_model()
    optimizer = torch.optim.AdamW(
        (p for p in dpo_model.parameters() if p.requires_grad), lr=lr,
    )
    n_pairs = len(pref_data["chosen_ids"])
    history = {"accuracy": [], "eval_reward": []}

    dpo_model.train()
    for step in range(n_steps):
        idx = np.random.choice(n_pairs, batch_size, replace=False)
        loss, info = dpo_loss(
            dpo_model,
            pref_data["chosen_ids"][idx],
            pref_data["rejected_ids"][idx],
            pref_data["mask"][idx],
            beta=beta,
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dpo_model.parameters(), 1.0)
        optimizer.step()

        history["accuracy"].append(info["accuracy"])

        if (step + 1) % 50 == 0:
            dpo_model.eval()
            ep_ids, ep_mask = encode_prompts(prompts[:16])
            fids = generate_responses(dpo_model, ep_ids, ep_mask)
            er = compute_sentiment_reward(decode_texts(fids)).mean().item()
            history["eval_reward"].append(er)
            dpo_model.train()
            print(f"  Step {step+1:3d}/{n_steps} | acc {info['accuracy']:.3f} "
                  f"| eval reward {er:.3f}")

    return history, dpo_model


dpo_history, dpo_model = train_dpo(pref_data)
dpo_reward = evaluate_model(dpo_model, prompts, label="DPO")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
w = 20
data = np.array(dpo_history["accuracy"])
axes[0].plot(data, alpha=0.25, color="coral")
if len(data) >= w:
    sm = np.convolve(data, np.ones(w) / w, mode="valid")
    axes[0].plot(range(w - 1, len(data)), sm, color="coral", lw=2)
axes[0].set_xlabel("Step"); axes[0].set_ylabel("Preference Accuracy")
axes[0].set_title("DPO Preference Accuracy")

er = dpo_history["eval_reward"]
axes[1].plot(range(len(er)), er, "o-", color="seagreen", lw=2)
axes[1].set_xlabel("Evaluation checkpoint"); axes[1].set_ylabel("Sentiment Reward")
axes[1].set_title("DPO Eval Reward")
fig.suptitle("DPO Training", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

### 💭 Discussion Question

**Question:** DPO needs no reward model and no online sampling. What are the trade-offs vs PPO-KL?


**Answer:**

---

# Part 4: Comparative Evaluation & Discussion (10 min)

In [ ]:
print("=" * 60)
print(f"  {'Method':<14} {'Sentiment Reward':>18}")
print("=" * 60)
for name, reward in [("Base GPT-2", base_reward),
                     ("SFT (ref)", sft_reward),
                     ("PPO-KL", ppo_reward),
                     ("DPO", dpo_reward)]:
    bar_len = int((reward + 1) / 2 * 30)  # map [-1,1] → [0,30]
    bar = "#" * max(bar_len, 0)
    print(f"  {name:<14} {reward:>+14.3f}    {bar}")
print("=" * 60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
w = 10

ax = axes[0]
for label, hist, color in [
    ("PPO-KL", ppo_history, "steelblue"),
]:
    data = np.array(hist["reward"])
    if len(data) >= w:
        sm = np.convolve(data, np.ones(w) / w, mode="valid")
        ax.plot(range(w - 1, len(data)), sm, label=label, color=color, lw=2)
if dpo_history["eval_reward"]:
    er = dpo_history["eval_reward"]
    xs = np.linspace(0, len(ppo_history["reward"]) - 1, len(er))
    ax.plot(xs, er, "o--", label="DPO (eval)", color="coral", lw=2)
ax.set_xlabel("Iteration"); ax.set_ylabel("Sentiment Reward")
ax.set_title("Reward During Training"); ax.legend()

ax = axes[1]
for label, hist, color in [
    ("PPO-KL", ppo_history, "steelblue"),
]:
    data = np.array(hist["kl"])
    if len(data) >= w:
        sm = np.convolve(data, np.ones(w) / w, mode="valid")
        ax.plot(range(w - 1, len(data)), sm, label=label, color=color, lw=2)
ax.set_xlabel("Iteration"); ax.set_ylabel("KL from Reference")
ax.set_title("KL Divergence"); ax.legend()

plt.tight_layout(); plt.show()

In [ ]:
print("Side-by-side generation comparison")
print("=" * 70)
for test_prompt in prompts[:3]:
    print(f"\nPrompt: \"{test_prompt}\"\n")
    p_ids, p_mask = encode_prompts([test_prompt])
    for name, m in [("PPO-KL", ppo_model), ("DPO", dpo_model)]:
        m.eval()
        fids = generate_responses(m, p_ids, p_mask)
        text = decode_texts(fids)[0]
        r = compute_sentiment_reward([text]).item()
        print(f"  [{name:>6s}  r={r:.2f}]  {text[:160]}")
    print("-" * 70)

### 💭 Discussion Question

**Question:** Which method achieves the best reward-per-unit-of-KL? Why does this metric matter for real-world RLHF?


**Answer:**

---

# Summary

## Key Takeaways

### 1. Supervised Fine-Tuning (SFT)
- Fine-tune the base LM on demonstration data to produce $\pi_{\text{SFT}}$
- Serves as the **starting point** and **reference policy** for all RL methods

### 2. PPO with KL Control
- Standard RLHF Stage III: maximise $r_\phi(x,y) - \beta\,\text{KL}[\pi_\theta \| \pi_{\text{ref}}]$
- Uses a clipped surrogate objective with importance sampling
- Requires a reward model, a value network, and online generation

### 3. Direct Preference Optimization (DPO)
- Derives a closed-form loss from the KL-constrained reward maximisation problem
- $\mathcal{L}_{\text{DPO}} = -\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$
- **No reward model, no online sampling, no value network**

### Comparison at a Glance

| | PPO-KL | DPO |
|---|---|---|
| Reward Model | Yes | No |
| Value Network | Yes | No |
| Online Sampling | Yes | No |
| Preference Data | Not directly | Yes |
| Memory Cost | High | Low |

---

## 🎯 Bonus Challenges (Optional)

1. **Adaptive KL:** Implement adaptive $\beta$ that increases when KL exceeds a target and decreases otherwise
2. **IPO:** Implement the Identity Preference Optimisation loss and compare with DPO
3. **Reward hacking:** Add noise to the sentiment classifier and observe which method is most robust

---

## References

- Schulman et al. (2017). *Proximal Policy Optimization Algorithms*. arXiv:1707.06347
- Ouyang et al. (2022). *Training language models to follow instructions with human feedback*. NeurIPS
- Rafailov et al. (2023). *Direct Preference Optimization*. NeurIPS

---

**Congratulations!** You've completed the RLHF optimization exercises.